# Telco Customer Churn — Exploratory Data Analysis

Goals of this notebook:
1. Load and inspect the raw IBM Telco Customer Churn dataset.
2. Check for data-quality issues (e.g. the `TotalCharges` blank-string bug).
3. Visualize the churn distribution and quantify class imbalance.
4. Visualize how key features (contract type, tenure, monthly charges, internet service) relate to churn.

These findings directly motivate the choices made in `src/preprocessing.py` (missing-value handling, SMOTE) and `src/train.py` (evaluation metrics beyond accuracy).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

DATA_PATH = "../data/telco_churn.csv"
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

## 1. Basic structure and dtypes

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

## 2. Data quality: the `TotalCharges` issue

`TotalCharges` is stored as an `object` (string) column even though it's numeric. A small number of rows contain a blank string `" "` instead of a number — these turn out to be customers with `tenure == 0` (brand new accounts that haven't been billed yet).

In [ ]:
print("dtype:", df["TotalCharges"].dtype)

blank_mask = df["TotalCharges"].str.strip() == ""
print(f"Blank TotalCharges rows: {blank_mask.sum()}")
df.loc[blank_mask, ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]]

In [ ]:
# Confirms the hypothesis: every blank TotalCharges row has tenure == 0.
df.loc[blank_mask, "tenure"].unique()

**Conclusion:** coerce `TotalCharges` to numeric and fill the resulting NaNs with `0`, since that's the true billed amount for a customer who just signed up. This is implemented in `src/preprocessing.py::load_data`.

Also worth noting: no other column in this dataset has missing values, and there are no duplicate `customerID`s.

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print("Remaining nulls after coercion:\n", df.isnull().sum()[df.isnull().sum() > 0])
print("\nDuplicate customerIDs:", df["customerID"].duplicated().sum())
df["TotalCharges"] = df["TotalCharges"].fillna(0.0)

## 3. Churn distribution and class imbalance

In [ ]:
churn_counts = df["Churn"].value_counts()
churn_rate = df["Churn"].value_counts(normalize=True)
print(churn_counts)
print(churn_rate)

fig, ax = plt.subplots(figsize=(5, 4))
sns.countplot(data=df, x="Churn", hue="Churn", palette="Set2", legend=False, ax=ax)
ax.set_title(f"Churn distribution (churn rate = {churn_rate['Yes']:.1%})")
plt.tight_layout()
plt.show()

About **26.5% of customers churn**, a roughly 1:2.8 minority/majority split. This is a meaningful class imbalance:
- A model that always predicts "No churn" would score ~73.5% accuracy while catching **zero** actual churners.
- This motivates (a) using precision/recall/F1/ROC-AUC instead of accuracy as the primary metrics, and (b) applying SMOTE to the training split so models see a balanced signal during learning.


## 4. Key feature relationships with churn

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
churn_by_contract = pd.crosstab(df["Contract"], df["Churn"], normalize="index")
churn_by_contract["Yes"].sort_values().plot(kind="barh", ax=ax, color="#d84c4c")
ax.set_xlabel("Churn rate")
ax.set_title("Churn rate by contract type")
plt.tight_layout()
plt.show()

Month-to-month customers churn at a dramatically higher rate than one- or two-year contract holders — contract length is one of the strongest churn signals in this dataset.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.kdeplot(data=df, x="tenure", hue="Churn", fill=True, common_norm=False, alpha=0.4, ax=ax)
ax.set_title("Tenure distribution by churn status")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.kdeplot(data=df, x="MonthlyCharges", hue="Churn", fill=True, common_norm=False, alpha=0.4, ax=ax)
ax.set_title("Monthly charges distribution by churn status")
plt.tight_layout()
plt.show()

Churners skew toward **low tenure** (they leave early) and **higher monthly charges** (price sensitivity / perceived value).

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
churn_by_internet = pd.crosstab(df["InternetService"], df["Churn"], normalize="index")
churn_by_internet["Yes"].sort_values().plot(kind="barh", ax=ax, color="#2f6fed")
ax.set_xlabel("Churn rate")
ax.set_title("Churn rate by internet service type")
plt.tight_layout()
plt.show()

Fiber optic customers churn far more than DSL or no-internet customers — plausibly linked to fiber's higher price point and/or service reliability complaints in this dataset.

In [ ]:
numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
corr = df[numeric_cols + []].copy()
corr["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(corr.corr(), annot=True, cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation: numeric features vs. churn")
plt.tight_layout()
plt.show()

## 5. Summary

- **Data quality:** `TotalCharges` needs numeric coercion + fill for 11 blank-string rows (tenure-0 customers). No other missing values; no duplicate customer IDs.
- **Class imbalance:** ~26.5% churn rate — accuracy alone would be misleading; use precision/recall/F1/ROC-AUC and apply SMOTE on the training split only.
- **Strongest churn signals observed:** short contract length (month-to-month), low tenure, higher monthly charges, and fiber-optic internet service. These are consistent with the SHAP feature-importance results produced later in `src/explain.py`.